# Oxford Physics College Analysis

Statistical comparison of Oxford colleges for Physics (F303) applicants.
Methodology: 3-year rolling shortlist rates, Wilson score confidence intervals,
Bayesian shrinkage toward the Oxford-wide mean, capacity signal (places per applicant).

**Data window**: Application rounds 2022, 2023, 2024 (entry years 2023, 2024, 2025).

**Primary data source**: Oxford Annual Admissions Statistical Report 2024 (PDF, open in browser).
URL: https://www.ox.ac.uk/sites/files/oxford/AnnualAdmissionsStatisticalReport2024.pdf

**Known caveat**: Shortlisting for Physics is department-led, not per-college.
Reallocation corrects for oversubscription before interviews. Per-college raw rates
reflect applicant self-selection + pre-reallocation capacity + noise.

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import beta as beta_dist
from IPython.display import display

pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:.3f}'.format)

## 1. Load data

In [ ]:
df = pd.read_csv('../data/college_physics_raw.csv')

# Coerce numeric columns — guards against stray strings from CSV misalignment
numeric_cols = [
    'applicants_2022', 'shortlisted_2022', 'places_2022',
    'applicants_2023', 'shortlisted_2023', 'places_2023',
    'applicants_2024', 'shortlisted_2024', 'places_2024',
    'typical_places_per_year',
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# University-wide shortlist totals for cross-checking (from Oxford Physics Dept reports)
UNIV_TOTALS = {
    2022: {'applicants': 1633, 'shortlisted': 492, 'places': 190},
    2023: {'applicants': 1672, 'shortlisted': 536, 'places': 187},
    2024: {'applicants': 1790, 'shortlisted': 525, 'places': 188},
}

# Oxford-wide 3-year shortlist rate (used as prior mean for Bayesian shrinkage)
TOTAL_APPS = sum(v['applicants'] for v in UNIV_TOTALS.values())
TOTAL_SL   = sum(v['shortlisted'] for v in UNIV_TOTALS.values())
PRIOR_MEAN = TOTAL_SL / TOTAL_APPS
print(f'University-wide 3-year shortlist rate (prior mean): {PRIOR_MEAN:.1%}')
print(f'  ({TOTAL_SL} shortlisted from {TOTAL_APPS} applicants, 2022-2024)')

df.head()

## 2. Aggregate and basic rates

In [ ]:
# 3-year totals
df['n'] = (df['applicants_2022'].fillna(0) +
           df['applicants_2023'].fillna(0) +
           df['applicants_2024'].fillna(0))

df['k'] = (df['shortlisted_2022'].fillna(0) +
           df['shortlisted_2023'].fillna(0) +
           df['shortlisted_2024'].fillna(0))

df['raw_rate'] = np.where(df['n'] > 0, df['k'] / df['n'], np.nan)

# Suppress colleges with very few applicants
df['suppressed'] = df['n'] < 15

print(f"Colleges with data: {(df['n'] > 0).sum()} / {len(df)}")
print(f"\nUniversity totals check:")
print(f"  Sum of college n: {df['n'].sum():.0f}  (expected: {TOTAL_APPS})")
print(f"  Sum of college k: {df['k'].sum():.0f}  (expected: {TOTAL_SL})")
print("  (totals will only match once all colleges are populated)")

## 3. Wilson score confidence intervals

Wilson intervals are used (not Wald) because they behave correctly for small n
and never produce impossible negative lower bounds.

In [ ]:
def wilson_ci(k, n, z=1.96):
    """95% Wilson score interval for proportion k/n."""
    if n == 0 or np.isnan(n) or np.isnan(k):
        return (np.nan, np.nan)
    p_hat = k / n
    denom = 1 + z**2 / n
    centre = (p_hat + z**2 / (2 * n)) / denom
    margin = (z * np.sqrt(p_hat * (1 - p_hat) / n + z**2 / (4 * n**2))) / denom
    return (centre - margin, centre + margin)

df[['wilson_lo', 'wilson_hi']] = df.apply(
    lambda r: wilson_ci(r['k'], r['n']), axis=1, result_type='expand'
)

# Sanity check: interval must contain observed rate
valid = df[df['n'] > 0]
assert (valid['wilson_lo'] <= valid['raw_rate'] + 1e-9).all(), 'Wilson lo > raw_rate'
assert (valid['wilson_hi'] >= valid['raw_rate'] - 1e-9).all(), 'Wilson hi < raw_rate'
print('Wilson interval sanity check passed.')

## 4. Bayesian shrinkage

Prior: Beta(alpha_0, beta_0) equivalent to `K_PSEUDO` observations at `PRIOR_MEAN`.

Colleges with few applicants (n < ~20) get pulled strongly toward the Oxford mean.
Colleges with many applicants (n > ~200) are barely moved.

In [ ]:
K_PSEUDO = 20   # equivalent prior sample size — explicit modelling choice
alpha_0 = PRIOR_MEAN * K_PSEUDO
beta_0  = (1 - PRIOR_MEAN) * K_PSEUDO

print(f'Prior: Beta({alpha_0:.2f}, {beta_0:.2f})')
print(f'  = {K_PSEUDO} prior observations at {PRIOR_MEAN:.1%}')
print(f'  College with n=20 is pulled ~50% toward prior mean')
print(f'  College with n=200 is pulled ~9% toward prior mean')

# Posterior mean
df['shrunk_rate'] = np.where(
    df['n'] > 0,
    (df['k'] + alpha_0) / (df['n'] + K_PSEUDO),
    np.nan
)

# Posterior 95% credible interval via Beta quantiles
def bayes_ci(k, n):
    if np.isnan(k) or np.isnan(n) or n == 0:
        return (np.nan, np.nan)
    a = k + alpha_0
    b = n - k + beta_0
    return (beta_dist.ppf(0.025, a, b), beta_dist.ppf(0.975, a, b))

df[['bayes_lo', 'bayes_hi']] = df.apply(
    lambda r: bayes_ci(r['k'], r['n']), axis=1, result_type='expand'
)

# Sanity check: shrinkage direction
valid = df[df['n'] > 0].copy()
above_prior = valid[valid['raw_rate'] > PRIOR_MEAN]
below_prior = valid[valid['raw_rate'] < PRIOR_MEAN]
if len(above_prior) > 0:
    assert (above_prior['shrunk_rate'] <= above_prior['raw_rate'] + 1e-9).all(), 'Shrinkage wrong direction (above)'
if len(below_prior) > 0:
    assert (below_prior['shrunk_rate'] >= below_prior['raw_rate'] - 1e-9).all(), 'Shrinkage wrong direction (below)'
print('Bayesian shrinkage direction check passed.')

## 5. Capacity signal

In [ ]:
df['mean_apps_per_year'] = np.where(df['n'] > 0, df['n'] / 3, np.nan)
df['places_per_applicant'] = np.where(
    df['typical_places_per_year'].notna() & (df['mean_apps_per_year'] > 0),
    df['typical_places_per_year'] / df['mean_apps_per_year'],
    np.nan
)

## 6. Ranking and AVOID flag

In [ ]:
valid_idx = df['n'] > 0

df['rank_raw']    = df['raw_rate'].rank(ascending=False, method='min', na_option='bottom')
df['rank_shrunk'] = df['shrunk_rate'].rank(ascending=False, method='min', na_option='bottom')
df['rank_change'] = df['rank_raw'] - df['rank_shrunk']

# AVOID: shrunk rate is meaningfully below Oxford mean AND CI upper bound is also below mean
# Only flag where we have enough data (not suppressed)
AVOID_THRESHOLD = PRIOR_MEAN - 0.05   # 5pp below the Oxford mean shortlist rate

df['AVOID'] = (
    (df['shrunk_rate'] < AVOID_THRESHOLD) &
    (df['wilson_hi'] < PRIOR_MEAN) &
    (~df['suppressed'])
)

print(f'Oxford-wide prior mean: {PRIOR_MEAN:.1%}')
print(f'AVOID threshold (shrunk rate): < {AVOID_THRESHOLD:.1%}')
print(f'Colleges flagged AVOID: {df["AVOID"].sum()}')
print(f'Suppressed colleges (n<15): {df["suppressed"].sum()}')

## 7. Results table

In [ ]:
out = df[df['n'] > 0].copy().sort_values('rank_shrunk')

out['raw_%']     = out['raw_rate'].map(lambda x: f'{x:.1%}' if pd.notna(x) else 'n/a')
out['shrunk_%']  = out['shrunk_rate'].map(lambda x: f'{x:.1%}' if pd.notna(x) else 'n/a')
out['wilson_95'] = out.apply(
    lambda r: f"[{r['wilson_lo']:.1%}, {r['wilson_hi']:.1%}]"
              if pd.notna(r['wilson_lo']) else 'n/a', axis=1
)
out['bayes_95']  = out.apply(
    lambda r: f"[{r['bayes_lo']:.1%}, {r['bayes_hi']:.1%}]"
              if pd.notna(r['bayes_lo']) else 'n/a', axis=1
)
out['ppa']       = out['places_per_applicant'].map(
    lambda x: f'{x:.3f}' if pd.notna(x) else '?'
)
out['flag']      = out['AVOID'].map({True: 'AVOID', False: ''})
out['flag']     += out['suppressed'].map({True: ' (thin)', False: ''})

display_cols = [
    'college', 'n', 'k',
    'raw_%', 'wilson_95',
    'shrunk_%', 'bayes_95',
    'typical_places_per_year', 'ppa',
    'rank_raw', 'rank_shrunk', 'rank_change',
    'flag'
]

display(out[display_cols].reset_index(drop=True))

## 8. Decomposition notes for AVOID colleges

In [ ]:
avoid_df = df[df['AVOID']]

if len(avoid_df) == 0:
    print('No colleges meet the AVOID criteria.')
    print(f'  (Threshold: shrunk rate < {AVOID_THRESHOLD:.1%} AND Wilson CI upper < {PRIOR_MEAN:.1%})')
    print('  Inspect the table above for any colleges with consistently below-average shrunk rates.')
else:
    median_ppa = df['places_per_applicant'].median()
    for _, row in avoid_df.iterrows():
        capacity_below = (pd.notna(row['places_per_applicant']) and
                          row['places_per_applicant'] < median_ppa)
        residual = row['shrunk_rate'] - PRIOR_MEAN
        print(f"\n{'='*60}")
        print(f"AVOID: {row['college']}")
        print(f"  3-year aggregate: {row['k']:.0f} shortlisted / {row['n']:.0f} applicants")
        print(f"  Raw shortlist rate:   {row['raw_rate']:.1%}")
        print(f"  Shrunk rate:          {row['shrunk_rate']:.1%}  (Oxford mean: {PRIOR_MEAN:.1%})")
        print(f"  Wilson 95% CI:        [{row['wilson_lo']:.1%}, {row['wilson_hi']:.1%}]")
        print(f"  Residual shortfall:   {residual:.1%} below Oxford mean (after shrinkage)")
        print()
        print(f"  Decomposition:")
        print(f"  [Capacity effect]     {'YES — below-median places/applicant ratio' if capacity_below else 'NO — places/applicant at or above median'}")
        print(f"  [Applicant-pool]      Unknown without conditional PAT score data.")
        print(f"                        If high-quality applicants self-select here, the raw")
        print(f"                        shortlist rate underestimates the college's generosity.")
        print(f"  [Reallocation]        Partially corrects pre-interview oversubscription.")
        print(f"                        Once reallocated, shortlisted applicants face the same offer decision.")
        print(f"  [Residual]            {residual:.1%} unexplained after shrinkage. Within noise unless")
        print(f"                        replicated across all three individual years.")

## 9. Verification checks

In [ ]:
print('=== Verification checks ===')
print()

# Check 1: University totals
sum_n = df['n'].sum()
sum_k = df['k'].sum()
tol_n = abs(sum_n - TOTAL_APPS) / TOTAL_APPS
tol_k = abs(sum_k - TOTAL_SL)   / TOTAL_SL
print(f'[1] University total check:')
print(f'    Sum of college applicants: {sum_n:.0f}  (expected {TOTAL_APPS}, diff {tol_n:.1%})')
print(f'    Sum of college shortlisted: {sum_k:.0f}  (expected {TOTAL_SL}, diff {tol_k:.1%})')
if tol_n > 0.05 or tol_k > 0.05:
    print('    WARNING: >5% discrepancy — check for missing or duplicate colleges')
else:
    print('    OK')

print()

# Check 2: Keble cross-check (confirmed from 3 separate PDFs)
keble = df[df['college'] == 'Keble'].iloc[0]
expected_keble = {'n': 388, 'k': 88}
print(f'[2] Keble cross-check (expected n=388, k=88 from Keble feedback PDFs):')
print(f'    n={keble["n"]:.0f}, k={keble["k"]:.0f}')
if abs(keble['n'] - expected_keble['n']) > 2:
    print('    WARNING: Keble applicant count does not match PDF source')
else:
    print('    OK')

print()

# Check 3: Wilson CI widths for thin colleges
thin = df[(df['n'] > 0) & (df['n'] < 40)]
if len(thin) > 0:
    widths = (thin['wilson_hi'] - thin['wilson_lo'])
    narrow = thin[widths < 0.20]
    if len(narrow) > 0:
        print(f'[3] WARNING: {len(narrow)} thin college(s) have suspiciously narrow Wilson intervals (<20pp):')
        print(narrow[['college', 'n', 'wilson_lo', 'wilson_hi']].to_string())
    else:
        print(f'[3] Wilson CI width check: OK (all thin colleges n<40 have CI width ≥20pp)')
else:
    print('[3] No thin colleges to check.')

print()

# Check 4: AVOID count sanity
avoid_count = df['AVOID'].sum()
print(f'[4] AVOID flag count: {avoid_count}')
if avoid_count == 0:
    print('    Note: 0 colleges flagged. Inspect shrunk rates directly for below-average outliers.')
elif avoid_count > 5:
    print('    WARNING: >5 colleges flagged. Threshold may be too permissive.')
else:
    print('    OK')

## 10. Save outputs

In [ ]:
import os
os.makedirs('../output', exist_ok=True)

# Full CSV
df.to_csv('../output/college_physics_analysis.csv', index=False)
print('Saved: ../output/college_physics_analysis.csv')

# Compute per-year Keble rates for the summary
keble_row = df[df['college'] == 'Keble'].iloc[0]
keble_yr = {
    2022: (keble_row['shortlisted_2022'], keble_row['applicants_2022']),
    2023: (keble_row['shortlisted_2023'], keble_row['applicants_2023']),
    2024: (keble_row['shortlisted_2024'], keble_row['applicants_2024']),
}

# Keble capacity: applicants per place (3yr average)
keble_apps_per_place = (keble_row['n'] / 3) / keble_row['typical_places_per_year']
oxford_apps_per_place = (TOTAL_APPS / 3) / 183  # 183 = Physics-only faculty places

colleges_populated = int((df['n'] > 0).sum())
colleges_total = len(df)

lines = [
    '# Oxford Physics College Analysis — Summary',
    '',
    f'Generated from confirmed data. **{colleges_populated} of {colleges_total} colleges populated.**',
    f'Remaining {colleges_total - colleges_populated} colleges need data from the Annual Admissions Statistical Report.',
    '',
    '---',
    '',
    '## Data sources and status',
    '',
    f'**Oxford-wide shortlist rate (prior mean)**: {PRIOR_MEAN:.1%}',
    f'  Confirmed from Physics Department annual reports: 492/1633 (2022), 536/1672 (2023), 525/1790 (2024).',
    '',
    '**Per-college data confirmed from college-specific PDFs:**',
    '',
    '| College | 2022 apps/SL | 2023 apps/SL | 2024 apps/SL | 3yr rate | Shrunk rate | Bayesian 95% CI |',
    '|---------|-------------|-------------|-------------|----------|-------------|-----------------|',
    (f"| Keble   | {keble_yr[2022][1]:.0f}/{keble_yr[2022][0]:.0f} ({keble_yr[2022][0]/keble_yr[2022][1]:.1%}) "
     f"| {keble_yr[2023][1]:.0f}/{keble_yr[2023][0]:.0f} ({keble_yr[2023][0]/keble_yr[2023][1]:.1%}) "
     f"| {keble_yr[2024][1]:.0f}/{keble_yr[2024][0]:.0f} ({keble_yr[2024][0]/keble_yr[2024][1]:.1%}) "
     f"| {keble_row['raw_rate']:.1%} "
     f"| {keble_row['shrunk_rate']:.1%} "
     f"| [{keble_row['bayes_lo']:.1%}, {keble_row['bayes_hi']:.1%}] |"),
    '',
    '**Intake numbers confirmed from college websites** (capacity signal, partial):',
    '',
    '| College | Places/yr | Avg apps/yr | Apps per place |',
    '|---------|-----------|-------------|----------------|',
]

intake_confirmed = df[df['typical_places_per_year'].notna() & (df['n'] == 0)].copy()
intake_confirmed['avg_apps_est'] = intake_confirmed['typical_places_per_year'] / PRIOR_MEAN * 2.5 / 3
for _, row in intake_confirmed.sort_values('college').iterrows():
    lines.append(
        f"| {row['college']} | {row['typical_places_per_year']:.0f} | "
        f"(needs shortlist data) | (needs shortlist data) |"
    )
# Add Keble with actual data
lines.append(
    f"| Keble | {keble_row['typical_places_per_year']:.0f} "
    f"| {keble_row['n']/3:.0f} "
    f"| {keble_apps_per_place:.1f} vs Oxford avg {oxford_apps_per_place:.1f} |"
)

lines += [
    '',
    '---',
    '',
    '## Key finding: Keble (confirmed)',
    '',
    f'Keble has a **{PRIOR_MEAN - keble_row["raw_rate"]:.1%} shortfall** below the Oxford mean in raw shortlist rate.',
    f'After Bayesian shrinkage (K=20 prior observations at {PRIOR_MEAN:.1%}), the shortfall is **{PRIOR_MEAN - keble_row["shrunk_rate"]:.1%}**.',
    f'The Wilson 95% CI upper bound [{keble_row["wilson_hi"]:.1%}] is still below the Oxford mean [{PRIOR_MEAN:.1%}],',
    'making Keble **statistically distinguishable as below average** even after accounting for sampling uncertainty.',
    '',
    '**Per-year breakdown** (confirms this is not a single-year anomaly):',
    '',
    '| Year | Applicants | Shortlisted | Rate | vs Oxford mean |',
    '|------|-----------|------------|------|----------------|',
]

for yr, (sl, apps) in keble_yr.items():
    univ_rate = UNIV_TOTALS[yr]['shortlisted'] / UNIV_TOTALS[yr]['applicants']
    rate = sl / apps
    diff = rate - univ_rate
    lines.append(
        f"| {yr} | {apps:.0f} | {sl:.0f} | {rate:.1%} | {diff:+.1%} vs {univ_rate:.1%} |"
    )

lines += [
    '',
    '**Capacity interpretation:**',
    f'Keble averages {keble_row["n"]/3:.0f} applicants/year for {keble_row["typical_places_per_year"]:.0f} places',
    f'= **{keble_apps_per_place:.1f} applicants per place**, vs Oxford-wide average of {oxford_apps_per_place:.1f}.',
    'Keble is attracting significantly more applicants per available place than the typical college.',
    'Reallocation partially corrects this (Keble shortlisted 35/30/23 applicants across 3 years, reallocating',
    'some OUT to other colleges), but the raw shortlist rate remains well below average.',
    '',
    '**Decomposition:**',
    '- Capacity effect: YES — ~{:.1f}x the Oxford-average applicants per place'.format(keble_apps_per_place / oxford_apps_per_place),
    '- Applicant-pool effect: Unknown without conditional PAT scores.',
    '  If Keble attracts stronger-than-average applicants, the shortlist rate would be higher than',
    '  expected — but the data shows the opposite, suggesting capacity is the dominant effect.',
    '- Reallocation effect: Partially corrects oversubscription. Post-reallocation, shortlisted',
    '  applicants face the same offer process regardless of college.',
    '- Residual: -7.4% unexplained by the prior alone, replicated across all 3 years.',
    '',
    '**Verdict on Keble**: Meets AVOID criteria. Applicants to Keble historically face a lower',
    'probability of being shortlisted than the Oxford average. Reallocation may save some, but',
    'it is a pre-interview correction only — applicants who are not shortlisted are not rescued.',
    '',
    '---',
    '',
    '## What is still needed to complete the ranking',
    '',
    'The per-college shortlisting data for the remaining 29 colleges is in the Oxford Annual',
    'Admissions Statistical Report (PDF). This document is publicly available but requires',
    'a browser session to download (programmatic access returns HTTP 403).',
    '',
    '**To complete the analysis, open this URL in your browser:**',
    'https://www.ox.ac.uk/sites/files/oxford/AnnualAdmissionsStatisticalReport2024.pdf',
    '',
    'Navigate to the Physics section. Find the table titled "Applications and admissions',
    'by college and course 2022–2024". For each college, record:',
    '- applicants_2022 / _2023 / _2024',
    '- shortlisted_2022 / _2023 / _2024 (or offers — note which in the `notes` column)',
    '',
    'Enter the data in: `data/college_physics_raw.csv`',
    '',
    '**Keble cross-check**: The report should show ~141/120/127 applicants and ~35/30/23 shortlisted.',
    'If it shows different numbers, note whether the metric is "shortlisted" or "offered".',
    '',
    '**Typical places/year** for colleges without intake data: visit each college Physics page.',
    'Known: Balliol≈8, Keble=8, St Hugh\'s=6, University≈10, Wadham=8.',
    '',
    '---',
    '',
    '## Provisional bottom-line (Keble data only)',
    '',
    'Based on confirmed data, **Keble should be treated with caution** due to historically',
    f'high applicant-to-place ratios ({keble_apps_per_place:.1f}x Oxford average).',
    '',
    'The full ranking across all 30 colleges cannot be completed without the Annual Report data.',
    'Most colleges will likely cluster near the 30.5% Oxford mean after shrinkage.',
    '',
    '**Working decision rule (provisional)**:',
    '- Avoid Keble unless other factors strongly favour it.',
    '- Treat all other colleges as statistically indistinguishable until data is entered.',
    '- ESAT score dominates college choice for any college not demonstrably worse than the mean.',
    '- Reallocation provides a safety net but not full compensation.',
    '',
    '## Key caveats',
    '',
    '1. This analysis uses shortlisted-to-applicants rates, not offer rates. The shortlisting step',
    '   is where college choice matters most; reallocation then partially equalises.',
    '2. Applicant-pool composition (self-selection by stronger candidates) cannot be assessed',
    '   without conditional PAT score data by college.',
    '3. Prior strength K=20 is a modelling choice — changing to K=10 or K=30 shifts shrunk',
    '   rates slightly but does not change the rank ordering for well-populated colleges like Keble.',
    '4. Harris Manchester does not admit Physics undergraduates (excluded from dataset).',
]

with open('../output/college_physics_summary.md', 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))

print('Saved: ../output/college_physics_summary.md')
print()
print('=== CONFIRMED FINDING ===')
print(f'Keble: 3yr shortlist rate {keble_row["raw_rate"]:.1%} (shrunk: {keble_row["shrunk_rate"]:.1%})')
print(f'Oxford mean: {PRIOR_MEAN:.1%}')
print(f'Gap: {PRIOR_MEAN - keble_row["shrunk_rate"]:.1%} below mean | Wilson CI upper: {keble_row["wilson_hi"]:.1%} (still below mean)')
print(f'Applicants/place: {keble_apps_per_place:.1f} vs Oxford avg {oxford_apps_per_place:.1f}')
print(f'Per-year rates: 2022={keble_yr[2022][0]/keble_yr[2022][1]:.1%}, 2023={keble_yr[2023][0]/keble_yr[2023][1]:.1%}, 2024={keble_yr[2024][0]/keble_yr[2024][1]:.1%}')
print(f'AVOID flag: {keble_row["AVOID"]}')

---
## Data entry guide

To populate the remaining colleges, open the Annual Admissions Statistical Report in your browser:

**URL**: https://www.ox.ac.uk/sites/files/oxford/AnnualAdmissionsStatisticalReport2024.pdf

Navigate to the Physics / Natural Sciences section. You will find a table titled something like
"Applications and admissions by college and course 2022–2024".

For each college, record:
- **applicants_2022 / _2023 / _2024**: applications received at that college
- **shortlisted_2022 / _2023 / _2024**: candidates shortlisted (or offered — note which in the `notes` column)
- **places_2022 / _2023 / _2024**: places filled (if shown)
- **typical_places_per_year**: from the college's own Physics page

**Keble cross-check**: Keble should show ~141/120/127 applicants and ~35/30/23 shortlisted.
If the Annual Report uses different numbers, the metric being tracked is different (e.g. offers not shortlisted).

Once the CSV is updated, re-run this notebook from the top.